In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install lifelines -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 10.4 MB/s eta 0:00:00


In [3]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold

Define paths and constants

In [4]:
PROCESSED_DATA_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/"
SELECTKBEST_OUTPUT_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/SelectKBest/"
MODEL_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Model/SelectKBest/"
RESULTS_DIR = "/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/SelectKBest/"

os.makedirs(SELECTKBEST_OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_PATH = SELECTKBEST_OUTPUT_DIR + "hcc_selectkbest_train_selected.csv"
TEST_PATH = SELECTKBEST_OUTPUT_DIR + "hcc_selectkbest_test_selected.csv"

MODEL_PATH = MODEL_DIR + "hcc_selectkbest_coxph_model.pkl"
CV_RESULTS_PATH = RESULTS_DIR + "hcc_selectkbest_coxph_cv_results.csv"
CV_DETAILS_PATH = RESULTS_DIR + "hcc_selectkbest_coxph_cv_fold_details.csv"

DURATION_COL = "OS_MONTHS"
EVENT_COL = "OS_EVENT"

PENALIZER_GRID = [0.01, 0.05, 0.1, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
L1_RATIO_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]

N_SPLITS = 5
RANDOM_STATE = 42

Load selected train and test data

In [5]:
train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)

train_data.shape, test_data.shape


((1085, 52), (272, 52))

In [6]:
test_data.head()

,SAMPLE_COVERAGE,TUMOR_SIZE,MUT_ALB,MUT_ARID1A,MUT_ARID1B,MUT_UPF1,MUT_WHSC1,MUT_ZFHX3,CNA_CDKN2A,CNA_KMT2B,...,RESECTION_MARGIN_R0,NEOADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_No,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_,PALLIATIVE_SYSTEMIC_THERPAY_No,PALLIATIVE_SYSTEMIC_THERPAY_Yes,OS_MONTHS,OS_EVENT
0,913.0,6.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,28.980,1
1,560.0,13.7,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,5.952,1
2,592.0,4.5,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,9.984,0
3,642.0,6.7,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,23.652,1
4,634.0,4.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,36.696,0


Basic checks

In [7]:
print("Train missing values:", int(train_data.isna().sum().sum()))
print("Test missing values:", int(test_data.isna().sum().sum()))

print("\nTrain OS_EVENT distribution:")
print(train_data[EVENT_COL].value_counts())

print("\nTest OS_EVENT distribution:")
print(test_data[EVENT_COL].value_counts())

Train missing values: 0
Test missing values: 0

Train OS_EVENT distribution:
OS_EVENT
1    641
0    444
Name: count, dtype: int64

Test OS_EVENT distribution:
OS_EVENT
1    161
0    111
Name: count, dtype: int64


ID leakage guard

In [8]:
for id_col in ["PATIENT_ID", "SAMPLE_ID"]:
    assert id_col not in train_data.columns, f"{id_col} found in train_data"
    assert id_col not in test_data.columns, f"{id_col} found in test_data"

print("ID leakage guard passed.")


ID leakage guard passed.


Optional drop block

In [9]:
COLS_TO_DROP = []

if COLS_TO_DROP:
    train_data = train_data.drop(columns=COLS_TO_DROP)
    test_data = test_data.drop(columns=COLS_TO_DROP)

train_data.shape, test_data.shape


((1085, 52), (272, 52))

Stratified CV helper

In [10]:
def apply_cross_validation(df, penalty, l1_ratio, duration_col, event_col, n_splits=5, random_state=42):
    X = df.reset_index(drop=True)
    y = X[event_col].astype(int).values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_details = []
    c_index_scores = []

    for fold_idx, (train_index, val_index) in enumerate(skf.split(X, y), start=1):
        train_fold = X.iloc[train_index].copy()
        val_fold = X.iloc[val_index].copy()

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                cph = CoxPHFitter(penalizer=penalty, l1_ratio=l1_ratio)
                cph.fit(train_fold, duration_col=duration_col, event_col=event_col)

            val_predictions = cph.predict_partial_hazard(val_fold)
            c_index = concordance_index(
                val_fold[duration_col],
                -val_predictions,
                val_fold[event_col]
            )

            c_index_scores.append(float(c_index))
            fold_details.append({
                "penalizer": penalty,
                "l1_ratio": l1_ratio,
                "fold": fold_idx,
                "status": "success",
                "c_index": float(c_index),
                "error_message": ""
            })

        except Exception as e:
            fold_details.append({
                "penalizer": penalty,
                "l1_ratio": l1_ratio,
                "fold": fold_idx,
                "status": "failed",
                "c_index": np.nan,
                "error_message": str(e)
            })

    return c_index_scores, fold_details


Grid search

In [11]:
cv_results = []
cv_fold_details = []

for penalty in PENALIZER_GRID:
    for l1_ratio in L1_RATIO_GRID:
        scores, details = apply_cross_validation(
            df=train_data,
            penalty=penalty,
            l1_ratio=l1_ratio,
            duration_col=DURATION_COL,
            event_col=EVENT_COL,
            n_splits=N_SPLITS,
            random_state=RANDOM_STATE
        )

        cv_fold_details.extend(details)

        successful_folds = len(scores)
        failed_folds = N_SPLITS - successful_folds

        cv_results.append({
            "penalizer": penalty,
            "l1_ratio": l1_ratio,
            "successful_folds": successful_folds,
            "failed_folds": failed_folds,
            "mean_c_index": np.mean(scores) if successful_folds > 0 else np.nan,
            "std_c_index": np.std(scores) if successful_folds > 0 else np.nan,
            "status": "ok" if successful_folds > 0 else "all_failed"
        })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    ["mean_c_index", "successful_folds"],
    ascending=[False, False]
).reset_index(drop=True)

cv_fold_details_df = pd.DataFrame(cv_fold_details)

cv_results_df.head()


,penalizer,l1_ratio,successful_folds,failed_folds,mean_c_index,std_c_index,status
0,0.05,0.25,5,0,0.725084,0.007185,ok
1,0.01,0.75,5,0,0.724250,0.009377,ok
2,0.30,0.00,5,0,0.724113,0.007587,ok
3,0.01,0.50,5,0,0.723983,0.009720,ok
4,0.50,0.00,5,0,0.723904,0.006607,ok


Save CV diagnostics

In [12]:
cv_results_df.to_csv(CV_RESULTS_PATH, index=False)
cv_fold_details_df.to_csv(CV_DETAILS_PATH, index=False)

CV_RESULTS_PATH, CV_DETAILS_PATH


('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/SelectKBest/hcc_selectkbest_coxph_cv_results.csv',
 '/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Results/SelectKBest/hcc_selectkbest_coxph_cv_fold_details.csv')

Select best setting

In [13]:
valid_results = cv_results_df.dropna(subset=["mean_c_index"]).copy()
best_penalty = valid_results.loc[0, "penalizer"]
best_l1_ratio = valid_results.loc[0, "l1_ratio"]
best_cv_c_index = valid_results.loc[0, "mean_c_index"]

print("Best penalizer:", best_penalty)
print("Best l1_ratio:", best_l1_ratio)
print("Best mean validation C-index:", round(best_cv_c_index, 4))


Best penalizer: 0.05
Best l1_ratio: 0.25
Best mean validation C-index: 0.7251


Train final model

In [14]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cph_final = CoxPHFitter(penalizer=best_penalty, l1_ratio=best_l1_ratio)
    cph_final.fit(train_data, duration_col=DURATION_COL, event_col=EVENT_COL)

cph_final.print_summary()


<lifelines.CoxPHFitter: fitted with 1085 total observations, 444 right-censored observations>
             duration col = 'OS_MONTHS'
                event col = 'OS_EVENT'
                penalizer = 0.05
                 l1 ratio = 0.25
      baseline estimation = breslow
   number of observations = 1085
number of events observed = 641
   partial log-likelihood = -3737.54
         time fit was run = 2026-07-02 12:09:35 UTC

---
                                          coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                 
SAMPLE_COVERAGE                           0.00      1.00      0.00           -0.00            0.00                1.00                1.00
TUMOR_SIZE                                0.01      1.01      0.01           -0.01            0.04                0.99                1.04
MUT_ALB                                  -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
MUT_ARID1A                                0.14      1.16      0.10           -0.06            0.35                0.94                1.42
MUT_ARID1B                               -0.19      0.83      0.29           -0.77            0.38                0.46                1.47
MUT_UPF1                                 -0.03      0.97      0.65           -1.31            1.25                0.27                3.48
MUT_WHSC1                                -0.80      0.45      0.59           -1.95            0.35                0.14                1.42
MUT_ZFHX3                                -0.90      0.41      0.28           -1.45           -0.35                0.23                0.71
CNA_CDKN2A                               -0.19      0.83      0.16           -0.51            0.13                0.60                1.14
CNA_KMT2B                                -0.73      0.48      0.42           -1.55            0.09                0.21                1.09
CNA_CDKN2Ap16INK4A                       -0.03      0.97      0.17           -0.36            0.29                0.70                1.34
SAMPLE_TYPE_Metastasis                   -0.15      0.86      0.09           -0.33            0.04                0.72                1.04
SAMPLE_TYPE_Primary                       0.00      1.00      0.00           -0.00            0.00                1.00                1.00
METASTATIC_SITE_Not Applicable           -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
PRIMARY_SITE_Liver                       -0.17      0.84      0.09           -0.36            0.01                0.70                1.01
GENE_PANEL_IMPACT341                     -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
GENE_PANEL_IMPACT410                      0.02      1.02      0.09           -0.16            0.20                0.85                1.22
GENE_PANEL_IMPACT468                      0.00      1.00      0.00           -0.00            0.00                1.00                1.00
GENE_PANEL_IMPACT505                     -0.22      0.80      0.14           -0.50            0.06                0.61                1.06
PRED_CATEGORY_Biliary class               0.42      1.52      0.13            0.17            0.68                1.18                1.97
HISTO_SUBTYPE_                           -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
ETHNICITY_Non-Spanish; Non-Hispanic       0.12      1.12      0.16           -0.19            0.43                0.83                1.53
ETHNICITY_Unknown                        -0.00      1.00      0.00           -0.01            0.01                0.99                1.01
ETHNICITY_Unknown whether Spanish or not -0.13      0.88      0.36           -0.84         

Training C-index

In [15]:
train_c_index = cph_final.concordance_index_
print(f"Training Concordance Index: {train_c_index:.4f}")


Training Concordance Index: 0.7374


Quick test check

In [16]:
test_predictions = cph_final.predict_partial_hazard(test_data)
test_c_index = concordance_index(
    test_data[DURATION_COL],
    -test_predictions,
    test_data[EVENT_COL]
)

print(f"Test Concordance Index: {test_c_index:.4f}")


Test Concordance Index: 0.7401


Coefficient summary

In [17]:
cox_summary = cph_final.summary[["coef", "exp(coef)", "p"]].sort_values("p")
cox_summary.head(15)


,coef,exp(coef),p
covariate,,,
PRED_CATEGORY_Biliary class,0.421749,1.524626,0.001217
MUT_ZFHX3,-0.899333,0.406841,0.001385
RESECTION_MARGIN_R0,-0.326134,0.721708,0.021128
METASTASIS_Yes,0.463949,1.590342,0.030902
PRIMARY_SITE_Liver,-0.174948,0.839501,0.060045
CNA_KMT2B,-0.732991,0.480470,0.079897
P_N_Negative,-0.163841,0.848877,0.086629
SAMPLE_TYPE_Metastasis,-0.148551,0.861956,0.115248
GENE_PANEL_IMPACT505,-0.217173,0.804790,0.126703


Save model

In [18]:
with open(MODEL_PATH, "wb") as pickle_file:
    pickle.dump(cph_final, pickle_file)

MODEL_PATH


'/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Model/SelectKBest/hcc_selectkbest_coxph_model.pkl'

**SelectKBest CoxPH Summary**

The updated `SelectKBest` CoxPH training notebook completed successfully using the 50-feature selected dataset. Cross-validation searched both the Cox regularization strength and mixing parameter, and the best configuration was `penalizer = 0.05` with `l1_ratio = 0.25`. That combination achieved a best mean validation C-index of `0.7251`, which is slightly stronger than the corresponding `exp_1` CoxPH validation result and suggests that the compact SelectKBest feature space supports a well-regularized Cox model.

The final trained model remained stable between train and test. Training concordance was `0.7374` and test concordance was `0.7401`, which is a very healthy pattern. The close alignment between the two indicates that the model generalized well to the held-out cohort and did not show obvious overfitting despite using a broader 50-feature input set.

At the model level, the CoxPH fit was strong overall. The fitted model used `1085` training observations with `641` observed events, achieved model concordance of `0.74`, and showed a strong likelihood ratio test (`306.73` on `50` degrees of freedom, `-log2(p) = 125.79`). Taken together, these statistics indicate that the fitted survival model is carrying meaningful prognostic information rather than behaving like a weak or unstable baseline.

The feature effects were also clinically and biologically interpretable. Higher risk was associated most strongly with `METASTASIS_Yes`, `C_N_Yes`, and `PRED_CATEGORY_Biliary class`, while lower risk was associated with `RESECTION_MARGIN_R0`, `CNA_CDKN2Ap16INK4A`, `MUT_ZFHX3`, and `PRIMARY_SITE_Liver`. Additional variables such as `PALLIATIVE_SYSTEMIC_THERPAY_No`, `ADJUVANT_SYSTEMIC_THERAPY_Yes`, `P_N_Negative`, `MUT_ARID1A`, `PNI_`, and `TUMOR_SIZE` also appeared among the more influential terms, though with weaker statistical support.

Overall, this notebook shows that the `SelectKBest` branch is a strong CoxPH baseline. The reduced top-50 feature set preserved enough survival signal to produce slightly improved held-out performance relative to `exp_1`, while also keeping the model interpretable and computationally efficient. This gives us a solid classical benchmark for the `SelectKBest` branch before we compare it against the updated DeepSurv results.